# COVID-19 Global Pandemic — Data Cleaning & Preparation
### Dataset: Our World in Data (OWID) COVID-19 Dataset
---
**Project type:** Data Cleaning & EDA Preparation  
**Dataset source:** [Our World in Data — COVID-19](https://github.com/owid/covid-19-data)  
**Notebook version:** 1.0  
**Status:** Portfolio / Academic Submission


## 1. Research Understanding

The OWID COVID-19 dataset is one of the most comprehensive public records of the pandemic. It tracks confirmed cases, deaths, testing, hospitalizations, vaccinations, policy responses, and country-level demographic indicators — all in one longitudinal panel.

Cleaning this dataset is not straightforward. It mixes fundamentally different types of variables:

- **Time-series metrics** that evolve daily (new cases, ICU patients, vaccination doses)
- **Cumulative counters** that should never decrease (total cases, total vaccinations)
- **Derived/smoothed metrics** computed from raw series (7-day averages, per-million rates)
- **Static country indicators** that are fixed for each country and don't change over time (GDP per capita, median age, life expectancy)

Each type demands a different cleaning strategy. Applying a single blanket imputation rule across all columns is one of the most common mistakes in this dataset — and this notebook avoids that.

**Primary cleaning objectives:**
- Ensure correct data types for every column
- Apply contextually appropriate missing-value strategies per column type
- Preserve meaningful nulls rather than fabricating data
- Flag coverage limitations and data quality issues for downstream users
- Produce a clean, analysis-ready dataset without distorting any signal


## 2. Dataset Overview

The OWID COVID-19 dataset is updated regularly and covers most countries from early 2020 onward. It integrates data from multiple upstream sources — the WHO, Johns Hopkins, national health ministries, Oxford's Government Response Tracker, and others.

**Key structural facts:**
- Each row is one country × one date observation
- Coverage is not uniform: richer countries with stronger surveillance systems have denser data
- Many columns have structural missingness (e.g., excess mortality exists only for ~60 countries)
- The dataset includes OWID-defined regional aggregates (e.g., "World", "Europe") alongside individual countries

**Column groups in this dataset:**

| Group | Examples |
|---|---|
| Identifiers | `iso_code`, `continent`, `location`, `date` |
| Confirmed cases | `total_cases`, `new_cases`, `new_cases_smoothed`, per-million variants |
| Confirmed deaths | `total_deaths`, `new_deaths`, `new_deaths_smoothed`, per-million variants |
| Excess mortality | `excess_mortality`, cumulative and per-million variants |
| Hospital & ICU | `icu_patients`, `hosp_patients`, weekly admissions, per-million variants |
| Policy responses | `stringency_index` |
| Reproduction rate | `reproduction_rate` |
| Testing | `total_tests`, `new_tests`, positivity rate, tests per case, `tests_units` |
| Vaccinations | `total_vaccinations`, `people_vaccinated`, `people_fully_vaccinated`, boosters, smoothed and per-hundred variants |
| Country indicators | population, demographics, economic indicators, health system capacity |


## 3. Data Dictionary — Key Cleaning Notes

Before touching the data, it helps to understand what each column *is*. The cleaning strategy follows directly from the variable type.

**Variable types in this dataset:**

| Type | Behavior | Cleaning implication |
|---|---|---|
| Cumulative counter | Monotonically non-decreasing | Forward-fill within country, then `cummax()` to fix dips |
| Daily flow | Can be zero, should not be negative | Null-out negatives; leave NaN (don't fill with zero) |
| Smoothed metric | Derived from raw series; lags signal | Preserve nulls; never back-fill (introduces future leakage) |
| Per-unit normalized | Derived from raw ÷ population | Do not recompute unless necessary; do not fill with zero |
| Rate/proportion | Bounded 0–1 or 0–100 | Clip to valid range; forward-fill with caution |
| Static indicator | One value per country | Forward-fill + back-fill within country (value is the same across all rows) |
| Categorical string | Fixed vocabulary | Normalize casing, forward-fill within country |

**Coverage notes:**
- Excess mortality data exists for ~60 countries only. Nulls elsewhere are not errors.
- Hospital/ICU data was primarily reported by European and high-income countries.
- Testing data coverage improved over time but remains incomplete for many low-income countries.
- Vaccination data starts from late 2020 and has variable completeness.


## 4. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')


## 5. Load Data

Update `FILE_PATH` to point to your local copy of the OWID dataset.


In [ ]:
FILE_PATH = 'owid-covid-data.csv'

df_raw = pd.read_csv(FILE_PATH, low_memory=False)
df = df_raw.copy()   # always work on a copy; preserve the raw source

print(f"Rows: {df.shape[0]:,}    Columns: {df.shape[1]}")


## 6. Initial Data Inspection

Before cleaning anything, let's understand what we're working with — data types, a few sample rows, and basic descriptive statistics.


In [ ]:
df.head(3)


In [ ]:
df.info(verbose=True, show_counts=True)


In [ ]:
df.describe(include='all').T


## 7. Data Quality Assessment

Here we get a complete picture of missing values, duplicate rows, and date/identifier issues before we touch anything.


In [ ]:
# Missing value summary — sorted by % missing
missing = df.isnull().mean().mul(100).round(2).sort_values(ascending=False)
missing_df = missing[missing > 0].reset_index()
missing_df.columns = ['column', 'pct_missing']
print(missing_df.to_string(index=False))


In [ ]:
# Duplicate rows check
n_dupes = df.duplicated(subset=['location', 'date']).sum()
print(f"Duplicate (location, date) pairs: {n_dupes}")


In [ ]:
# OWID-defined aggregates (continents, income groups, etc.) — we'll remove these
owid_entries = df[df['iso_code'].str.startswith('OWID', na=False)]['location'].unique()
print(f"OWID aggregates to be removed ({len(owid_entries)}):")
print(owid_entries)


In [ ]:
# Date range coverage
print(f"Date range: {df['date'].min()} → {df['date'].max()}")
print(f"Unique countries (before filtering): {df['location'].nunique()}")


## 8. Data Cleaning

The cleaning is organized by column group, following the logic of each variable type. Read the notes above each block — they explain *why* a specific strategy was chosen, not just *what* the code does.


### 8.1 Identifiers & Metadata

These four columns form the backbone of the dataset. `iso_code`, `location`, and `continent` are strings; `date` needs to be a proper datetime. OWID regional aggregates (rows where `iso_code` starts with `OWID_`) are removed here — they're useful for some analyses but would distort country-level cleaning logic.


In [ ]:
# Normalize string identifiers
df['iso_code']  = df['iso_code'].astype(str).str.strip()
df['continent'] = df['continent'].astype(str).str.strip().str.title()
df['location']  = df['location'].astype(str).str.strip()

# Parse dates
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Remove OWID regional aggregates (continents, income groups, "World", etc.)
df = df[~df['iso_code'].str.startswith('OWID', na=False)].copy()

# Sort chronologically within each country — required for all subsequent ffill operations
df = df.sort_values(['location', 'date']).reset_index(drop=True)

print(f"Rows after removing OWID aggregates: {df.shape[0]:,}")
print(f"Countries remaining: {df['location'].nunique()}")
print(f"Date dtype: {df['date'].dtype}")


### 8.2 Confirmed Cases

**`total_cases`** is a cumulative counter — it should never decrease. Strategy: forward-fill gaps within each country (plausible because the count doesn't drop between report dates), then apply `cummax()` to correct any reported decreases.

**`new_cases`** is a daily flow. Negative values occur when countries issue retroactive corrections — they're set to null, not zero. Missing days are left as NaN (not filled with zero) because a true zero case day is meaningfully different from a day with no report.

**Smoothed and per-million variants** are derived metrics. We leave them largely as-is; the source already computed smoothing correctly. Filling smoothed columns with back-fill would introduce future data leakage.

> ⚠️ `total_cases_per_million` was previously filled with `0` — this is incorrect. A missing per-million value means the raw total is also missing, not that cases were zero. We leave it null.


In [ ]:
# --- total_cases (cumulative) ---
df['total_cases'] = pd.to_numeric(df['total_cases'], errors='coerce')
df['total_cases'] = df.groupby('location')['total_cases'].ffill()
df.loc[df['total_cases'] < 0, 'total_cases'] = np.nan
df['total_cases'] = df.groupby('location')['total_cases'].cummax()
# Note: rows where total_cases is still null are early pandemic rows with no reports.
# We keep them rather than drop — dropping would remove valid rows for other columns.

# --- new_cases (daily flow) ---
df['new_cases'] = pd.to_numeric(df['new_cases'], errors='coerce')
df.loc[df['new_cases'] < 0, 'new_cases'] = np.nan
# Leave NaN — do NOT fill with 0 (no-report ≠ zero cases)

# --- new_cases_smoothed (derived 7-day average) ---
df['new_cases_smoothed'] = pd.to_numeric(df['new_cases_smoothed'], errors='coerce')
df.loc[df['new_cases_smoothed'] < 0, 'new_cases_smoothed'] = np.nan
# Preserve nulls — back-filling a smoothed column introduces future leakage

# --- per-million variants (derived; leave nulls intact) ---
for col in ['total_cases_per_million', 'new_cases_per_million', 'new_cases_smoothed_per_million']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[df[col] < 0, col] = np.nan
    df[col] = df[col].round(3)

print("Cases columns cleaned.")
print(df[['total_cases', 'new_cases', 'new_cases_smoothed']].describe().T)


### 8.3 Confirmed Deaths

Same logic as cases. `total_deaths` is cumulative (forward-fill + cummax). `new_deaths` is a daily flow (null out negatives, leave gaps as NaN). `new_deaths_smoothed` is a derived series — preserve its nulls.

> ⚠️ The original notebook dropped all rows where `total_deaths` was null. This is too aggressive: it eliminates early pandemic rows for many countries where deaths hadn't started yet, but those rows still carry valid case, testing, and demographic data. We no longer do this.


In [ ]:
# --- total_deaths (cumulative) ---
df['total_deaths'] = pd.to_numeric(df['total_deaths'], errors='coerce')
df['total_deaths'] = df.groupby('location')['total_deaths'].ffill()
df.loc[df['total_deaths'] < 0, 'total_deaths'] = np.nan
df['total_deaths'] = df.groupby('location')['total_deaths'].cummax()

# --- new_deaths (daily flow) ---
df['new_deaths'] = pd.to_numeric(df['new_deaths'], errors='coerce')
df.loc[df['new_deaths'] < 0, 'new_deaths'] = np.nan
# Leave NaN — do NOT fill with 0

# --- new_deaths_smoothed (derived 7-day average) ---
df['new_deaths_smoothed'] = pd.to_numeric(df['new_deaths_smoothed'], errors='coerce')
df.loc[df['new_deaths_smoothed'] < 0, 'new_deaths_smoothed'] = np.nan
# Preserve nulls

# --- per-million variants ---
for col in ['total_deaths_per_million', 'new_deaths_per_million', 'new_deaths_smoothed_per_million']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[df[col] < 0, col] = np.nan
    df[col] = df[col].round(3)

print("Deaths columns cleaned.")
print(df[['total_deaths', 'new_deaths', 'new_deaths_smoothed']].describe().T)


### 8.4 Excess Mortality

Excess mortality is only available for countries that have reliable civil registration systems — roughly 60 countries. Nulls elsewhere are **structural** and expected. Filling them with zero would be statistically wrong (zero excess mortality is a real claim; null means "data does not exist").

These columns are reported at weekly or monthly frequency, so most rows within a reporting period are null by design.


In [ ]:
excess_cols = [
    'excess_mortality',
    'excess_mortality_cumulative',
    'excess_mortality_cumulative_absolute',
    'excess_mortality_cumulative_per_million'
]

for col in excess_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    # Do NOT fill with zero — structural nulls must stay null

coverage = df[excess_cols].notna().mean().mul(100).round(1)
print("Excess mortality coverage (% non-null rows):")
print(coverage.to_string())
print("\nNote: ~80%+ null is expected — coverage is limited to ~60 countries.")


### 8.5 Hospital & ICU

Hospital and ICU data was primarily reported by European countries. Coverage is sparse — expect high null rates globally. These are point-in-time daily counts (not cumulative), so forward-fill is defensible for short gaps (e.g., a country reports every other day), but we cap the fill at a reasonable window.

> ⚠️ The original notebook forward-filled these columns without any limit. Carrying a hospital census count forward for weeks or months during a period of actual no-reporting manufactures data. We use a limited fill window of 3 days.

> Note: `weekly_icu_admissions` and `weekly_hosp_admissions` are weekly totals, not daily — forward-filling across daily rows makes them appear as daily figures, which is misleading. We forward-fill but flag this for downstream users.


In [ ]:
hosp_point_cols = [
    'icu_patients', 'icu_patients_per_million',
    'hosp_patients', 'hosp_patients_per_million'
]

for col in hosp_point_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[df[col] < 0, col] = np.nan
    # Limited forward-fill (3 days) within each country
    df[col] = df.groupby('location')[col].ffill(limit=3)

hosp_weekly_cols = [
    'weekly_icu_admissions', 'weekly_icu_admissions_per_million',
    'weekly_hosp_admissions', 'weekly_hosp_admissions_per_million'
]

for col in hosp_weekly_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[df[col] < 0, col] = np.nan
    # Forward-fill up to 7 days (these are weekly reports)
    df[col] = df.groupby('location')[col].ffill(limit=7)

# Round per-million columns
for col in ['hosp_patients_per_million', 'icu_patients_per_million',
            'weekly_icu_admissions_per_million', 'weekly_hosp_admissions_per_million']:
    df[col] = df[col].round(3)

print("Hospital/ICU columns cleaned.")


### 8.6 Stringency Index

The Oxford Stringency Index is a composite measure (0–100) of government policy responses. It's updated when policies change, not daily — so forward-filling within a country is appropriate and reflects the real-world situation (policies stay in effect until changed).

Values outside [0, 100] are physically impossible and treated as encoding errors.


In [ ]:
df['stringency_index'] = pd.to_numeric(df['stringency_index'], errors='coerce')
df.loc[(df['stringency_index'] < 0) | (df['stringency_index'] > 100), 'stringency_index'] = np.nan
df['stringency_index'] = df.groupby('location')['stringency_index'].ffill()

print(df['stringency_index'].describe())


### 8.7 Reproduction Rate

The effective reproduction number (R) is a real-time modeled estimate, not a measured value. It can be above or below 1, and can legitimately be very high (e.g., at outbreak onset). Negative values are impossible and are treated as errors.

Forward-filling R is defensible for short gaps, but R can change fast during outbreaks. We use a limited fill window to avoid carrying stale estimates too far.


In [ ]:
df['reproduction_rate'] = pd.to_numeric(df['reproduction_rate'], errors='coerce')
df.loc[df['reproduction_rate'] < 0, 'reproduction_rate'] = np.nan
df['reproduction_rate'] = df.groupby('location')['reproduction_rate'].ffill(limit=7)

print(df['reproduction_rate'].describe())


### 8.8 Testing Columns

**`total_tests`** is a cumulative counter — same strategy as `total_cases`.

**`new_tests`** is a daily flow — null out negatives, leave gaps as NaN (do not fill with zero).

**`new_tests_smoothed`** is a 7-day derived metric — preserve its nulls, do not back-fill.

**`positive_rate`** is a proportion bounded [0, 1]. Values outside this range are encoding errors. Forward-fill is appropriate since this is a rolling average.

**`tests_per_case`** is the mathematical inverse of `positive_rate`. Very large values (e.g., Inf) occur when new cases approach zero; they should be nulled.

**`tests_units`** is a categorical string describing what was counted. Different countries use different definitions. We normalize casing, forward-fill within country (the unit doesn't change mid-series), and encode as category.

> ⚠️ The original notebook recomputed `new_tests_per_thousand` from scratch as `new_tests / population * 1000` and then forward-filled the result. This overwrites OWID's original values and introduces errors if `new_tests` had already been cleaned. We use OWID's pre-computed column directly.


In [ ]:
# --- total_tests (cumulative) ---
df['total_tests'] = pd.to_numeric(df['total_tests'], errors='coerce')
df['total_tests'] = df.groupby('location')['total_tests'].ffill()
df.loc[df['total_tests'] < 0, 'total_tests'] = np.nan
df['total_tests'] = df.groupby('location')['total_tests'].cummax()

# --- total_tests_per_thousand (cumulative, derived) ---
df['total_tests_per_thousand'] = pd.to_numeric(df['total_tests_per_thousand'], errors='coerce')
df['total_tests_per_thousand'] = df.groupby('location')['total_tests_per_thousand'].ffill()
df['total_tests_per_thousand'] = df.groupby('location')['total_tests_per_thousand'].cummax()
df.loc[df['total_tests_per_thousand'] < 0, 'total_tests_per_thousand'] = np.nan
df['total_tests_per_thousand'] = df['total_tests_per_thousand'].round(3)

# --- new_tests (daily flow) ---
df['new_tests'] = pd.to_numeric(df['new_tests'], errors='coerce')
df.loc[df['new_tests'] < 0, 'new_tests'] = np.nan
# Leave NaN — do NOT fill with 0

# --- new_tests_per_thousand (use OWID's column directly, do not recompute) ---
df['new_tests_per_thousand'] = pd.to_numeric(df['new_tests_per_thousand'], errors='coerce')
df.loc[df['new_tests_per_thousand'] < 0, 'new_tests_per_thousand'] = np.nan
df['new_tests_per_thousand'] = df['new_tests_per_thousand'].round(3)

# --- new_tests_smoothed (derived 7-day average) ---
df['new_tests_smoothed'] = pd.to_numeric(df['new_tests_smoothed'], errors='coerce')
df.loc[df['new_tests_smoothed'] < 0, 'new_tests_smoothed'] = np.nan
# Preserve nulls — do NOT back-fill

# --- new_tests_smoothed_per_thousand ---
df['new_tests_smoothed_per_thousand'] = pd.to_numeric(df['new_tests_smoothed_per_thousand'], errors='coerce')
df.loc[df['new_tests_smoothed_per_thousand'] < 0, 'new_tests_smoothed_per_thousand'] = np.nan
df['new_tests_smoothed_per_thousand'] = df['new_tests_smoothed_per_thousand'].round(3)

# --- positive_rate (proportion 0–1) ---
df['positive_rate'] = pd.to_numeric(df['positive_rate'], errors='coerce')
df.loc[(df['positive_rate'] < 0) | (df['positive_rate'] > 1), 'positive_rate'] = np.nan
df['positive_rate'] = df.groupby('location')['positive_rate'].ffill(limit=7)

# --- tests_per_case ---
df['tests_per_case'] = pd.to_numeric(df['tests_per_case'], errors='coerce')
df.loc[df['tests_per_case'] < 0, 'tests_per_case'] = np.nan
df.loc[df['tests_per_case'] == np.inf, 'tests_per_case'] = np.nan
df['tests_per_case'] = df.groupby('location')['tests_per_case'].ffill(limit=7)

# --- tests_units (categorical string) ---
valid_units = {'people tested', 'tests performed', 'samples tested'}
df['tests_units'] = df['tests_units'].astype(str).str.strip().str.lower()
df['tests_units'] = df['tests_units'].where(df['tests_units'].isin(valid_units), other=np.nan)
df['tests_units'] = df.groupby('location')['tests_units'].ffill()
df['tests_units'] = df['tests_units'].astype('category')

print("Testing columns cleaned.")
print(f"tests_units categories: {df['tests_units'].cat.categories.tolist()}")


### 8.9 Vaccination Columns

**Cumulative counters** (`total_vaccinations`, `people_vaccinated`, `people_fully_vaccinated`, `total_boosters`) follow the same strategy: forward-fill, then cummax. Countries often report vaccination totals every few days rather than daily, so gaps are common and forward-fill is appropriate.

**`new_vaccinations`** is a daily flow — null out negatives, leave gaps as NaN. The original notebook clipped this at the 99th percentile, which would distort actual surge data for large countries. We remove the clip.

**Smoothed and per-hundred/per-million variants** are derived metrics — leave nulls intact.

> ⚠️ The original notebook applied `fillna(0)` to `new_vaccinations_smoothed` directly, without first converting to numeric. That's a latent bug. We fix it here.


In [ ]:
# --- cumulative vaccination counters ---
cum_vax_cols = [
    'total_vaccinations',
    'people_vaccinated',
    'people_fully_vaccinated',
    'total_boosters'
]

for col in cum_vax_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df.groupby('location')[col].ffill()
    df.loc[df[col] < 0, col] = np.nan
    df[col] = df.groupby('location')[col].cummax()

# --- new_vaccinations (daily flow) ---
df['new_vaccinations'] = pd.to_numeric(df['new_vaccinations'], errors='coerce')
df.loc[df['new_vaccinations'] < 0, 'new_vaccinations'] = np.nan
# Do NOT clip at quantile — this distorts real surge data for large countries
# Do NOT fill with 0 — missing is not the same as zero

# --- new_vaccinations_smoothed (derived 7-day average) ---
df['new_vaccinations_smoothed'] = pd.to_numeric(df['new_vaccinations_smoothed'], errors='coerce')
# Leave nulls — do NOT fill with 0

# --- per-hundred cumulative variants ---
per_hundred_cols = [
    'total_vaccinations_per_hundred',
    'people_vaccinated_per_hundred',
    'people_fully_vaccinated_per_hundred',
    'total_boosters_per_hundred'
]
for col in per_hundred_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df.groupby('location')[col].ffill()
    df.loc[df[col] < 0, col] = np.nan
    df[col] = df[col].round(3)

# --- smoothed flow variants ---
smoothed_vax_cols = [
    'new_vaccinations_smoothed_per_million',
    'new_people_vaccinated_smoothed',
    'new_people_vaccinated_smoothed_per_hundred'
]
for col in smoothed_vax_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[df[col] < 0, col] = np.nan
    # Preserve nulls — derived columns

print("Vaccination columns cleaned.")
print(df[cum_vax_cols].describe().T)


### 8.10 Population & Static Country Indicators

These columns contain one value per country that doesn't change over time. The same number repeats across every row for a given country (or is null throughout). The correct approach:

1. Convert to numeric
2. Forward-fill *then* back-fill within each country (so the value propagates to every row)
3. Do **not** impute across countries — a missing value for one country cannot be inferred from another

Some of these indicators (e.g., `extreme_poverty`, `female_smokers`, `handwashing_facilities`) are missing for many countries — this reflects genuine data gaps in international statistics, not data entry errors.

> ⚠️ The original notebook applied a blanket `groupby('location').apply(lambda x: x.ffill().bfill())` at the very end, on all columns at once. This is dangerous: it accidentally forward-fills time-varying columns (like daily cases or ICU counts) using future values from the back-fill, and it processes all columns without any type awareness. We handle each group explicitly instead.


In [ ]:
# --- population ---
df['population'] = pd.to_numeric(df['population'], errors='coerce')
df.loc[df['population'] <= 0, 'population'] = np.nan
df['population'] = df.groupby('location')['population'].ffill().bfill()

# --- population_density ---
df['population_density'] = pd.to_numeric(df['population_density'], errors='coerce')
df['population_density'] = df.groupby('location')['population_density'].ffill().bfill()

# --- demographic indicators ---
demo_cols = ['median_age', 'aged_65_older', 'aged_70_older']
for col in demo_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df.groupby('location')[col].ffill().bfill()

# --- economic indicators ---
econ_cols = ['gdp_per_capita', 'extreme_poverty']
for col in econ_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df.groupby('location')[col].ffill().bfill()

# --- health system indicators ---
health_cols = [
    'cardiovasc_death_rate', 'diabetes_prevalence',
    'female_smokers', 'male_smokers',
    'handwashing_facilities', 'hospital_beds_per_thousand',
    'life_expectancy', 'human_development_index'
]
for col in health_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df.groupby('location')[col].ffill().bfill()

print("Static country indicators cleaned.")

# Coverage check — these should now have nulls only for countries absent from source data
coverage_check = df[['population'] + demo_cols + econ_cols + health_cols].isnull().mean().mul(100).round(1)
print("\nRemaining null % (after fill — structural gaps only):")
print(coverage_check.to_string())


## 9. Feature Engineering

A few derived features that are commonly useful for downstream analysis:


In [ ]:
# Case fatality rate — only compute where both values exist and total_cases > 0
df['case_fatality_rate'] = np.where(
    df['total_cases'] > 0,
    (df['total_deaths'] / df['total_cases']).round(4),
    np.nan
)

# Vaccination completion ratio — share of at-least-one-dose people who are fully vaccinated
df['vax_completion_ratio'] = np.where(
    df['people_vaccinated'] > 0,
    (df['people_fully_vaccinated'] / df['people_vaccinated']).round(4),
    np.nan
)

# Boolean flag for when testing data is available for a row
df['has_testing_data'] = df['total_tests'].notna()

print("Feature engineering complete.")
print(df[['case_fatality_rate', 'vax_completion_ratio', 'has_testing_data']].describe().T)


## 10. Cleaned Dataset Summary


In [ ]:
print("=== Final Dataset Shape ===")
print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

print("\n=== Date Range ===")
print(f"{df['date'].min().date()} → {df['date'].max().date()}")

print("\n=== Countries ===")
print(f"Unique countries: {df['location'].nunique()}")

print("\n=== Remaining Null % (top 20 columns by missingness) ===")
remaining_null = df.isnull().mean().mul(100).round(1).sort_values(ascending=False).head(20)
print(remaining_null.to_string())


In [ ]:
df.dtypes.to_frame('dtype').reset_index().rename(columns={'index': 'column'})


## 11. Key Insights & Data Quality Notes

A few things worth calling out before handing this dataset to analysis:

**Coverage is unequal by design.** High-income countries with strong public health infrastructure (US, UK, Germany, France) contribute dense, reliable data. Low-income countries, particularly in sub-Saharan Africa and parts of Asia, have sparse testing, hospital, and vaccination data. This isn't a data cleaning failure — it's a reflection of real-world reporting capacity.

**Excess mortality is the most incomplete metric.** Only ~60 countries have civil registration systems robust enough to generate these estimates. For the remaining ~130+ countries in this dataset, excess mortality is genuinely unknown — not zero.

**Smoothed columns are derivatives, not measurements.** `new_cases_smoothed`, `new_deaths_smoothed`, and their variants are 7-day rolling averages computed from the raw daily values. They should not be cleaned the same way as the raw counts they're derived from.

**Cumulative columns can have dips.** Countries occasionally revise their totals downward (e.g., after de-duplication or reclassification). The `cummax()` approach handles this conservatively, but in a few cases the true count may genuinely have decreased due to reclassification. Analysts should be aware of this.

**`tests_units` defines what counts as a "test."** Countries use different definitions (tests performed, people tested, samples tested). Comparisons of testing volume across countries with different units are not apples-to-apples.


## 12. Limitations of the Dataset

- **Reporting lag and data revisions:** Case and death counts are subject to retroactive revision. The dataset reflects the latest available version, but historical values may differ from earlier snapshots.
- **Definition changes over time:** Some countries changed their testing definitions or case counting methodology mid-pandemic. Smoothed series partially mitigate this, but breaks still exist.
- **Vaccination data stopped being updated** for some countries once the acute phase ended. Late-2022 and 2023 vaccination figures may be incomplete for certain regions.
- **Excess mortality models differ by source.** OWID uses the Karlinsky & Kobak (2021) methodology. Alternative models (e.g., The Economist, WHO) produce different estimates.
- **Reproduction rate estimates** are modeled outputs with uncertainty ranges that are not captured in this column.


## 13. Export Clean Dataset


In [ ]:
OUTPUT_PATH = 'owid_covid_cleaned.csv'
df.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned dataset saved to: {OUTPUT_PATH}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")


## 14. Recommendations for Next Steps

**For exploratory analysis:**
- Filter to countries with sufficient testing data before computing positivity rate trends
- Use `new_cases_smoothed` and `new_deaths_smoothed` for trend visualizations — the raw daily figures are noisy
- Weight country-level comparisons by population or use per-million metrics

**For modeling:**
- Static country indicators can be merged in once per country rather than repeated on every row
- Consider creating a lag feature for `reproduction_rate` — today's R predicts future cases
- Missing vaccination data in the early pandemic period is by definition (vaccines didn't exist), not a data quality issue — handle it as a time-conditional feature

**For reproducibility:**
- Pin the OWID dataset version if results need to be reproducible (the dataset is updated daily)
- Log the download date in your pipeline metadata
